In [ ]:
# [패키지 설치] OpenAI 호환 클라이언트 라이브러리 설치
# Ollama는 OpenAI API 규격을 그대로 사용하므로 openai 패키지로 통신 가능
pip install openai


In [ ]:
"""
Volvo Co-Pilot — 로컬 LLM 기반 굴착기 캐빈 내부 제어 시스템

[아키텍처 개요]
  - Edge (Raspberry Pi 4): 이 파일을 실행하는 주체
      * latest_command.txt 를 폴링(polling)하여 사용자 명령 감지
      * status.json 을 읽고 써서 하드웨어 상태 관리
      * 기상청 외부 API 호출 담당
  - Cloud (Google Colab T4 GPU): LLM 서버
      * Ollama로 llama3.1:8b 호스팅
      * bore 터널링을 통해 REST API 엔드포인트 노출
      * 자연어 이해(NLU) 및 Function Calling 처리 전담

[필요 패키지]
  pip install openai requests
"""

# ====== 표준 라이브러리 임포트 ======
import json                   # status.json 읽기/쓰기
from typing import Dict, Any  # 타입 힌트
from openai import OpenAI     # Ollama와 통신할 OpenAI 호환 클라이언트
import os                     # 파일 존재 여부 확인
import time                   # 폴링 간격(sleep) 제어
from pathlib import Path      # 크로스 플랫폼 파일 경로 처리
import csv                    # 대화 로그 CSV 저장
from datetime import datetime, timedelta  # 타임스탬프 및 날씨 날짜 계산
import uuid                   # 세션 고유 ID 생성
import sys                    # Jupyter 환경 감지
import requests               # 기상청 API HTTP 요청


# ====== LLM 클라이언트 초기화 (Ollama, bore 터널링) ======
# bore 터널링으로 노출된 Colab의 Ollama 서버에 접속.
# OpenAI 클라이언트를 그대로 사용하되, base_url만 로컬 Ollama 주소로 교체.
# api_key는 Ollama가 인증 없이 동작하므로 더미값 "ollama" 사용.
client = OpenAI(
    base_url="http://bore.pub:57685/v1",  # Colab bore 터널 주소 + /v1 필수
    api_key="ollama"                       # Ollama 인증 없음 → 더미값
)


# ====== 시스템 프롬프트 ======
# LLM이 매 요청마다 참조하는 역할 및 행동 지침.
# 소형 모델의 '도구 집착(Tool Obsession)'과 '문맥 오염(Context Poisoning)'을
# 방지하기 위해 OUT OF DOMAIN 규칙을 명시적으로 삽입.
SYSTEM_PROMPT = """You are a strict AI assistant inside an excavator cabin. 

[AVAILABLE TOOLS]
- AC Control: power_on, power_off, set_temperature, set_fan_speed, set_mode, set_swing
- Light Control: set_light
- Weather: get_weather

[CRITICAL RULES]
1. AC COMMANDS: If the user wants to control the air conditioner, YOU MUST use the AC Control tools.
2. LIGHT COMMANDS: If the user wants to turn on/off ANY light (e.g., "headlight", "LED", "rear light", "조명", "전조등", "후방등"), YOU MUST use the `set_light` tool.
3. WEATHER COMMANDS: If the user asks for the weather forecast, YOU MUST use the `get_weather` tool.
4. OUT OF DOMAIN (NO TOOLS!): For ALL other questions, greetings, or irrelevant topics, YOU MUST NOT call any tools. You must bypass tools and reply EXACTLY with this Korean sentence:
"I apologize, but I can only provide information related to excavator control and weather information."

Extra Rules:
- Prefer SI units (°C) for temperature. If only number is given, assume Celsius.
- Validate ranges: temperature 16–30°C; fan speed in ["low","medium","high","auto"]; mode in ["cool","heat","fan","auto","dry"]; swing in ["on","off"].
- If input is ambiguous (e.g., '좀 시원하게'), choose best defaults: mode='cool', fan='auto'.
- Combine multiple intents if present (e.g., '24도로 맞추고 바람은 약으로').
- Keep normal chat responses short.
"""


# ====== Function Calling 도구(TOOLS) 스키마 정의 ======
# LLM에 주입되는 JSON Schema 형태의 도구 목록.
# LLM은 사용자 입력을 분석한 뒤, 아래 스키마 중 하나를 선택해
# 'tool_calls' 형태로 응답하면 코드 측에서 실제 함수를 실행한다.
TOOLS = [
    # --- 에어컨 전원 켜기 (파라미터 없음) ---
    {
        "type": "function",
        "function": {
            "name": "power_on",
            "description": "Turn AC power on.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
    },
    # --- 에어컨 전원 끄기 (파라미터 없음) ---
    {
        "type": "function",
        "function": {
            "name": "power_off",
            "description": "Turn AC power off.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
    },
    # --- 온도 설정 (16~30°C, 범위 초과 시 handle_user에서 추가 검증) ---
    {
        "type": "function",
        "function": {
            "name": "set_temperature",
            "description": "Set target temperature in Celsius (16–30).",
            "parameters": {
                "type": "object",
                "properties": {"temp_c": {"type": "number", "minimum": 16, "maximum": 30}},
                "required": ["temp_c"],
                "additionalProperties": False,
            },
        },
    },
    # --- 팬(바람) 세기 설정 ---
    {
        "type": "function",
        "function": {
            "name": "set_fan_speed",
            "description": "Set fan speed.",
            "parameters": {
                "type": "object",
                "properties": {"level": {"type": "string", "enum": ["low","medium","high","auto"]}},
                "required": ["level"],
                "additionalProperties": False,
            },
        },
    },
    # --- 운전 모드 설정 (냉방/난방/송풍/자동/제습) ---
    {
        "type": "function",
        "function": {
            "name": "set_mode",
            "description": "Set AC mode.",
            "parameters": {
                "type": "object",
                "properties": {"mode": {"type": "string", "enum": ["cool","heat","fan","auto","dry"]}},
                "required": ["mode"],
                "additionalProperties": False,
            },
        },
    },
    # --- 루버 스윙(바람 방향 자동 조절) 설정 ---
    {
        "type": "function",
        "function": {
            "name": "set_swing",
            "description": "Set swing (louvers oscillation) on/off.",
            "parameters": {
                "type": "object",
                "properties": {"state": {"type": "string", "enum": ["on","off"]}},
                "required": ["state"],
                "additionalProperties": False,
            },
        },
    },
    # --- 날씨 조회 (기상청 단기예보 API 연동) ---
    # date: 'today' | 'tomorrow' 만 지원 (기상청 API 제약)
    # time: 시간대별 조회 (morning/afternoon/evening/night)
    # intent: 'summary'=전체 요약 / 'yesno'=비가 오나요? 형태의 예/아니오
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "description": "target date like 'today', 'tomorrow', 'yesterday', 'nextweek'"
                    },
                    "time": {
                        "type": "string",
                        "enum": ["morning", "afternoon", "evening", "night"]
                    },
                    "intent": {
                        "type": "string",
                        "enum": ["summary", "yesno"]
                    }
                },
                "required": ["date"]
            }
        }
    },
    # --- 조명 ON/OFF (전조등, LED, 후방등 등 모든 조명을 단일 키로 제어) ---
    {
        "type": "function",
        "function": {
            "name": "set_light",
            "description": "Turn the excavator's light (headlight, LED, rear light) on or off.",
            "parameters": {
                "type": "object",
                "properties": {
                    "state": {
                        "type": "string",
                        "enum": ["on", "off"],
                        "description": "Turn the light on or off"
                    }
                },
                "required": ["state"],
                "additionalProperties": False
            }
        }
    },
]


# ====== 하드웨어 상태 파일 경로 설정 ======
# Raspberry Pi 로컬에 저장되는 JSON 파일.
# 에어컨·조명 등 제어 결과가 이 파일에 실시간 반영되어
# 다른 하드웨어 프로세스(블루투스 컨트롤러 등)가 읽어 실제 장비를 구동한다.
STATUS_JSON_PATH = "/home/comfuture/Desktop/bluetooth/volvo-1/status.json"


def update_status_json(category: str, key: str, value: Any) -> bool:
    """
    status.json 파일을 안전하게 Read-Modify-Write 하는 핵심 유틸 함수.

    Args:
        category (str): 최상위 키 (예: "aux", "excavator")
        key      (str): 하위 키 (예: "ac_temperature", "headlight")
        value    (Any): 저장할 값 (숫자, 문자열 등)

    Returns:
        bool: 성공 True / 실패 False
    """
    try:
        # 1. 파일이 이미 존재하면 기존 데이터 유지하며 읽기
        if os.path.exists(STATUS_JSON_PATH):
            with open(STATUS_JSON_PATH, "r", encoding="utf-8") as f:
                status_data = json.load(f)
        else:
            # 파일이 없으면 기본 구조로 초기화
            status_data = {"aux": {}}

        # 2. category 키가 없으면 빈 딕셔너리로 생성 (예: "excavator" 카테고리 첫 사용 시)
        if category not in status_data:
            status_data[category] = {}

        # 3. 목표 키 값 업데이트 및 수정 시각 기록 (ISO8601, 초 단위)
        status_data[category][key] = value
        status_data["updated_at"] = datetime.now().isoformat(timespec="seconds")

        # 4. 전체 데이터를 파일에 덮어쓰기 (indent=2 로 가독성 유지)
        with open(STATUS_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(status_data, f, indent=2, ensure_ascii=False)

        return True
    except Exception as e:
        print(f"[ERROR] status.json 업데이트 실패: {e}")
        return False


# ====== 에어컨 로컬 콜백 함수 ======
# LLM이 tool_call을 내리면 execute_tool_call()이 아래 함수들을 호출한다.
# 현재는 콘솔 출력 + status.json 갱신만 수행하며,
# 실제 Volvo Co-Pilot SDK / 시리얼 제어 코드는 여기에 추가하면 된다.

def ac_power_on() -> bool:
    """에어컨 전원을 켠다. status.json의 aux.ac_power 를 1로 설정."""
    print("[AC] power ON")
    return update_status_json("aux", "ac_power", 1)  # 1: ON

def ac_power_off() -> bool:
    """에어컨 전원을 끈다. status.json의 aux.ac_power 를 0으로 설정."""
    print("[AC] power OFF")
    return update_status_json("aux", "ac_power", 0)  # 0: OFF

def ac_set_temperature(temp_c: float) -> bool:
    """설정 온도를 변경한다. 정수 변환 후 status.json 반영."""
    print(f"[AC] set temperature: {temp_c} °C")
    return update_status_json("aux", "ac_temperature", int(temp_c))

def ac_set_fan_speed(level: str) -> bool:
    """팬 세기를 변경한다. 문자열 레벨을 숫자 코드로 변환 후 저장."""
    print(f"[AC] set fan speed: {level}")
    # 하드웨어 프로토콜에 맞게 문자열 → 정수 코드 매핑
    speed_map = {"low": 1, "medium": 3, "high": 5, "auto": 0}
    val = speed_map.get(level, level)  # 매핑 실패 시 원본 문자열 그대로 저장
    return update_status_json("aux", "ac_fan_speed", val)

def ac_set_mode(mode: str) -> bool:
    """운전 모드를 변경한다 (cool/heat/fan/auto/dry)."""
    print(f"[AC] set mode: {mode}")
    return update_status_json("aux", "ac_mode", mode)

def ac_set_swing(state: str) -> bool:
    """루버 스윙(자동 바람 방향 조절)을 켜거나 끈다."""
    print(f"[AC] set swing: {state}")
    return update_status_json("aux", "ac_swing", state)

def set_light(state: str) -> bool:
    """전조등/LED 등 모든 조명을 켜거나 끈다. headlight 키로 통합 관리."""
    print(f"[LIGHT] headlight is set to {state}")
    return update_status_json("aux", "headlight", state)


# ====== 기상청 단기예보 API 설정 ======

# 실제 발급받은 기상청 API 서비스 키 (공공데이터포털)
KMA_SERVICE_KEY = "XDFWE2EF8u0+ZA6IOpdyKF7ugHMzrtDwlCGDzn0LQIiU0zHui/oRyAcyxR2hDthrlRlFW6Zo7IiElJw9Ugeguw=="

# LLM이 반환하는 시간대 문자열 → 기상청 API base_time 변환 테이블
# 기상청 단기예보는 3시간 간격으로 발표되므로 가장 가까운 기준 시간 사용
TIME_MAP = {
    "morning":   "0800",  # 오전
    "afternoon": "1400",  # 오후
    "evening":   "1700",  # 저녁
    "night":     "2000",  # 야간
}


def resolve_datetime(date_str: str, time_str: str | None):
    """
    기상청 API 호출에 필요한 base_date, base_time을 결정한다.

    ⚠️ 주의: 현재 base_date는 항상 '오늘' 날짜로 고정되어 있다.
    기상청 단기예보는 '발표 기준일'로 API를 호출하면 발표일 기준
    3일치 데이터를 모두 반환하기 때문에, 내일 데이터 필터링은
    confirmation_weather()에서 fcstDate 기준으로 처리한다.

    Args:
        date_str (str): LLM이 결정한 날짜 문자열 ('today', 'tomorrow' 등)
        time_str (str|None): 시간대 문자열 또는 None

    Returns:
        tuple[str, str]: (YYYYMMDD 형식의 base_date, HHMM 형식의 base_time)
    """
    now = datetime.now()
    base_date = now.strftime("%Y%m%d")              # 항상 오늘 기준으로 API 호출
    base_time = TIME_MAP.get(time_str, "1400")      # 시간대 미지정 시 오후 기본값
    return base_date, base_time


def get_weather(
    date: str,          # "today", "tomorrow" 또는 YYYYMMDD 형식
    time: str | None,   # "morning", "afternoon" 등 또는 None
    nx: int,            # 기상청 격자 X 좌표
    ny: int             # 기상청 격자 Y 좌표
) -> dict:
    """
    기상청 단기예보 API를 호출하여 원시 JSON 데이터를 반환한다.
    numOfRows=1000으로 충분한 예보 항목을 확보하여
    오늘/내일 데이터를 모두 포함하도록 한다.

    Returns:
        dict: 기상청 API 원본 응답 JSON
    """
    base_date, base_time = resolve_datetime(date, time)

    url = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
    params = {
        "serviceKey": KMA_SERVICE_KEY,
        "numOfRows":  1000,      # 오늘+내일 데이터 확보를 위해 넉넉하게 설정
        "pageNo":     1,
        "dataType":   "JSON",
        "base_date":  base_date,
        "base_time":  base_time,
        "nx":         nx,
        "ny":         ny
    }

    r = requests.get(url, params=params, timeout=5)
    r.raise_for_status()  # HTTP 4xx/5xx 시 예외 발생
    return r.json()


# ====== 날씨 조회 기본 위치 설정 ======
# 기상청 격자 좌표계(nx, ny) 사용. 변경 시 기상청 좌표 변환 도구 활용.
DEFAULT_LOCATION = {
    "name": "Busan",  # 부산 (현장 위치)
    "nx": 98,
    "ny": 76,
}


def fetch_weather(
    date: str = "today",
    time: str | None = None,
    location: dict = DEFAULT_LOCATION,
) -> dict:
    """
    get_weather()의 고수준 래퍼 함수.
    위치 정보(nx, ny)를 DEFAULT_LOCATION에서 자동으로 가져와
    handle_user()에서 간결하게 호출할 수 있도록 한다.

    Args:
        date     (str): 'today' | 'tomorrow'
        time     (str|None): 'morning' | 'afternoon' | 'evening' | 'night' | None
        location (dict): nx, ny 포함한 위치 딕셔너리

    Returns:
        dict: 기상청 API 원본 응답 JSON
    """
    base_date, base_time = resolve_datetime(date, time)
    return get_weather(
        date=base_date,
        time=base_time,
        nx=location["nx"],
        ny=location["ny"],
    )


# ====== 도구 실행 라우터 ======
# LLM이 선택한 tool_call name을 실제 Python 함수로 연결하는 디스패처.
# get_weather는 handle_user()에서 별도 처리되므로 여기서는 나머지 제어 함수만 담당.
def execute_tool_call(name: str, arguments: Dict[str, Any]) -> bool:
    """
    LLM이 결정한 도구 이름과 인자로 실제 장비 제어 함수를 호출한다.

    Args:
        name      (str): LLM이 선택한 tool_call 이름
        arguments (dict): LLM이 생성한 파라미터 딕셔너리

    Returns:
        bool: 실행 성공 여부
    """
    if name == "power_on":
        return ac_power_on()
    if name == "power_off":
        return ac_power_off()
    if name == "set_temperature":
        return ac_set_temperature(float(arguments["temp_c"]))
    if name == "set_fan_speed":
        return ac_set_fan_speed(arguments["level"])
    if name == "set_mode":
        return ac_set_mode(arguments["mode"])
    if name == "set_swing":
        return ac_set_swing(arguments["state"])
    if name == "set_light":
        return set_light(arguments["state"])
    if name == "get_weather":
        # ℹ️ 참고: get_weather는 handle_user()의 루프 안에서 직접 처리된다.
        # 이 분기는 만약 execute_tool_call이 단독 호출될 경우를 위한 안전망.
        return get_weather(
            arguments.get("date", "today"),
            arguments.get("time"),
            DEFAULT_LOCATION["nx"],
            DEFAULT_LOCATION["ny"]
        )
    print(f"[WARN] Unknown tool: {name}")
    return False


# ====== 한국어 확인 문장 생성기 ======
# 도구 실행 후 사용자에게 반환할 자연어 확인 문장을 생성한다.
# 실제 TTS(음성 출력)와 연동될 경우 이 문장이 발화된다.
def confirmation_korean(name: str, args: Dict[str, Any]) -> str:
    """
    tool_call 이름과 파라미터를 받아 한국어 확인 문장을 반환한다.

    Args:
        name (str): 실행된 도구 이름
        args (dict): 실행 시 사용된 파라미터

    Returns:
        str: 사용자에게 출력할 한국어 문장
    """
    if name == "power_on":
        return "에어컨 전원을 켰어요."
    if name == "power_off":
        return "에어컨 전원을 껐어요."
    if name == "set_temperature":
        return f"에어컨을 {int(args['temp_c'])}도로 맞췄어요."
    if name == "set_fan_speed":
        # 영어 레벨 → 한국어 표현 변환
        mapping = {"low": "약", "medium": "중", "high": "강", "auto": "자동"}
        return f"바람 세기를 {mapping.get(args['level'], args['level'])}으로 설정했어요."
    if name == "set_mode":
        # 영어 모드 → 한국어 표현 변환
        m = {"cool": "냉방", "heat": "난방", "fan": "송풍", "auto": "자동", "dry": "제습"}
        return f"모드를 {m.get(args['mode'], args['mode'])}으로 설정했어요."
    if name == "set_swing":
        return "에어 스윙을 켰어요." if args["state"] == "on" else "에어 스윙을 껐어요."
    if name == "set_light":
        target_state = "켰어요" if args["state"] == "on" else "껐어요"
        return f"조명을 {target_state}."
    # 위에 해당하지 않는 경우 범용 메시지
    return "설정을 적용했어요."


def confirmation_weather(weather_json: dict, intent: str = "summary", target_date: str = "today") -> str:
    """
    기상청 API 원본 응답 JSON을 파싱하여 사용자에게 전달할 날씨 문장을 반환한다.

    Args:
        weather_json (dict): get_weather() 또는 fetch_weather()가 반환한 원본 JSON
        intent       (str):  응답 형식 — 'summary'(전체 요약) | 'yesno'(강수 예/아니오)
        target_date  (str):  필터링할 날짜 — 'today' | 'tomorrow'

    Returns:
        str: 자연어 날씨 문장 (영어)
    """
    # 1. 기상청 응답 구조에서 예보 항목 리스트 추출
    try:
        items = weather_json["response"]["body"]["items"]["item"]
    except KeyError:
        return "날씨 정보를 불러오는 중 기상청 서버에서 문제가 발생했어요."

    # 2. 기상청 단기예보는 최대 3일치만 제공 → today/tomorrow 외에는 지원 불가
    if target_date not in ["today", "tomorrow"]:
        return f"Sorry, I can only provide weather forecasts for today and tomorrow. (You asked for '{target_date}')"

    # 3. target_date 문자열을 YYYYMMDD 형식으로 변환
    now = datetime.now()
    target_dt = now + timedelta(days=1) if target_date == "tomorrow" else now
    target_yyyymmdd = target_dt.strftime("%Y%m%d")

    # 4. API 응답에서 해당 날짜 항목만 필터링
    filtered_items = [i for i in items if i["fcstDate"] == target_yyyymmdd]

    # 5. 필터링 결과가 없으면 에러 반환 (오늘로 fallback 하지 않음)
    if not filtered_items:
        return "Sorry, there is no weather data available for the requested time."

    # 6. 필요한 기상 카테고리 값 추출
    # TMP: 기온(°C), SKY: 하늘 상태 코드, PTY: 강수 형태 코드
    temp = next((i["fcstValue"] for i in filtered_items if i["category"] == "TMP"), "unknown")
    sky  = next((i["fcstValue"] for i in filtered_items if i["category"] == "SKY"), "unknown")
    pty  = next((i["fcstValue"] for i in filtered_items if i["category"] == "PTY"), "unknown")

    # 기상청 코드 → 영어 문자열 변환 테이블
    sky_map = {"1": "Clear", "3": "Mostly cloudy", "4": "Cloudy"}
    pty_map = {"0": "none", "1": "rain", "2": "rain and snow", "3": "snow", "4": "shower"}

    # 7. intent에 따라 응답 형식 분기
    if intent == "yesno":
        # "비 오나요?" 형태의 단순 예/아니오 응답
        return "Yes, precipitation is expected." if pty != "0" else "No, precipitation is not expected."

    # 기본(summary): 기온 + 하늘 상태 + 강수 형태를 한 문장으로 요약
    pty_str = pty_map.get(pty, 'unknown')
    precip_desc = "no precipitation is expected" if pty_str == "none" else f"precipitation is {pty_str}"
    return f"The temperature is {temp}°C, sky condition is {sky_map.get(sky, 'unknown')}, and {precip_desc}."


# ====== 로깅 설정 ======
# 모든 대화 기록을 JSONL 및 CSV 형식으로 누적 저장한다.
# 졸업과제 평가 및 시스템 성능 분석에 활용.
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)  # logs/ 폴더 없으면 자동 생성

JSONL_PATH = LOG_DIR / "ac_controller_log.jsonl"  # JSON Lines: 한 줄 = 한 레코드
CSV_PATH   = LOG_DIR / "ac_controller_log.csv"    # CSV: 스프레드시트로 분석 용이

# 명령을 수신하는 텍스트 파일 경로 (음성인식/텍스트 입력 모듈이 이 파일에 쓴다)
COMMAND_FILE_PATH = "/home/comfuture/Desktop/capstone/volvo-1/LLMtoFunction/latest_command.txt"

# CSV 컬럼 헤더 정의
CSV_HEADERS = [
    "timestamp",        # ISO8601 타임스탬프
    "session_id",       # 실행 세션 고유 ID (프로세스마다 새로 발급)
    "user_text",        # 사용자가 입력한 자연어
    "normal_reply",     # 최종 응답 문장 (한국어 확인 또는 일반 대화)
    "tool_calls_json",  # LLM이 요청한 tool_call 목록 (JSON 직렬화)
    "exec_results_json" # 실제 로컬 콜백 실행 결과 목록 (JSON 직렬화)
]

# 노트북(또는 프로세스) 시작 시 한 번만 생성되는 세션 고유 ID
SESSION_ID = str(uuid.uuid4())


def _ensure_csv_header(path: Path):
    """CSV 파일이 없을 때만 헤더 행을 1회 기록한다."""
    if not path.exists():
        with path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(CSV_HEADERS)


def save_jsonl(record: dict):
    """대화 레코드를 JSONL 파일에 한 줄로 추가 저장한다 (UTF-8, 유니코드 그대로)."""
    with JSONL_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_csv(record: dict):
    """대화 레코드를 CSV 파일에 한 줄로 추가 저장한다. tool_calls/exec_results는 JSON 직렬화."""
    _ensure_csv_header(CSV_PATH)
    row = [
        record.get("timestamp", ""),
        record.get("session_id", ""),
        record.get("user_text", ""),
        record.get("normal_reply", ""),
        json.dumps(record.get("tool_calls", []), ensure_ascii=False),
        json.dumps(record.get("exec_results", []), ensure_ascii=False),
    ]
    with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(row)


# ====== 파일 폴링 유틸리티 ======

def read_last_nonempty_line(filepath: str) -> str | None:
    """
    텍스트 파일의 마지막 비어있지 않은 줄을 반환한다.
    음성 인식 모듈이 파일에 줄바꿈을 포함해 쓸 수 있으므로 역방향 탐색 사용.

    Returns:
        str | None: 유효한 마지막 줄 또는 None (파일 없음/빈 파일)
    """
    p = Path(filepath)
    if not p.exists() or not p.is_file():
        print(f"[WARN] 파일을 찾을 수 없습니다: {filepath}")
        return None
    try:
        lines = p.read_text(encoding="utf-8").splitlines()
        # 끝에서부터 역방향으로 탐색하여 첫 번째 비어있지 않은 줄 반환
        for line in reversed(lines):
            s = line.strip()
            if s:
                return s
        print(f"[WARN] 파일에 유효한 명령 행이 없습니다: {filepath}")
        return None
    except Exception as e:
        print(f"[ERROR] 파일 읽기 실패: {e}")
        return None


def read_latest_command_in_dir(dirpath: str) -> tuple[str, float, str] | None:
    """
    폴더 내에서 가장 최근에 수정된 .txt 파일을 찾아
    (파일 경로, mtime, 마지막 유효 줄) 튜플을 반환한다.
    디렉토리 폴링 모드에서 사용.

    Returns:
        tuple[str, float, str] | None: (경로, mtime, 명령 문자열) 또는 None
    """
    p = Path(dirpath)
    if not p.exists() or not p.is_dir():
        print(f"[WARN] 폴더를 찾을 수 없습니다: {dirpath}")
        return None
    txt_files = list(p.glob("*.txt"))
    if not txt_files:
        return None
    # 수정 시각 기준 내림차순 정렬 → 가장 최신 파일 선택
    txt_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)
    latest = txt_files[0]
    mtime = latest.stat().st_mtime
    cmd = read_last_nonempty_line(str(latest))
    if cmd is None:
        return None
    return (str(latest), mtime, cmd)


def loop_watch_directory(dirpath: str, interval_sec: float = 1.0):
    """
    [디렉토리 폴링 루프] (CLI --dir 모드용)
    폴더 내 가장 최근 수정된 .txt의 마지막 줄이 바뀔 때마다 handle_user()를 실행.
    mtime 또는 마지막 줄 중 하나라도 변경되면 재처리.

    Args:
        dirpath      (str): 감시할 폴더 경로
        interval_sec (float): 폴링 간격(초)
    """
    print(f"[INFO] 폴더 감시 시작: {dirpath} (interval={interval_sec}s)")
    last_seen: tuple[str, float, str] | None = None  # (경로, mtime, last_line)

    while True:
        info = read_latest_command_in_dir(dirpath)
        if info:
            path, mtime, last_line = info
            # 파일 경로, mtime, 내용 중 하나라도 변경되었을 때만 처리
            if (last_seen is None
                or path != last_seen[0]
                or mtime != last_seen[1]
                or last_line != last_seen[2]):
                print(f"\n> File : {path}")
                print(f"> User : {last_line}")
                print(f"< Bot  : {handle_user(last_line)}\n")
                last_seen = (path, mtime, last_line)
        time.sleep(interval_sec)


def loop_watch_file(filepath: str, interval_sec: float = 1.0):
    """
    [단일 파일 폴링 루프] (Jupyter 자동 실행 및 CLI --file 모드)
    지정 파일의 mtime 또는 마지막 줄이 변경될 때마다 handle_user()를 실행.
    같은 내용이 반복 기록되어도 mtime이 바뀌면 재처리.

    Args:
        filepath     (str): 감시할 텍스트 파일 경로
        interval_sec (float): 폴링 간격(초)
    """
    print(f"[INFO] 파일 감시 시작: {filepath} (interval={interval_sec}s)")
    last_seen: tuple[float, str] | None = None  # (mtime, last_line)
    while True:
        p = Path(filepath)
        if p.exists():
            try:
                mtime = p.stat().st_mtime
                last_line = read_last_nonempty_line(filepath)
                if last_line:
                    # mtime 또는 내용이 달라졌을 때만 LLM 처리 실행
                    if last_seen is None or (mtime != last_seen[0] or last_line != last_seen[1]):
                        print(f"\n> File : {filepath}")
                        print(f"> User : {last_line}")
                        print(f"< Bot  : {handle_user(last_line)}\n")
                        last_seen = (mtime, last_line)
            except Exception as e:
                print(f"[ERROR] 감시 중 오류: {e}")
        time.sleep(interval_sec)


# ====== 대화 메모리 (슬라이딩 윈도우) ======
# 소형 LLM의 컨텍스트 창 한계 및 과거 날씨 질문에 갇히는 문제(Context Poisoning)를 방지하기 위해
# 최근 MAX_MEMORY개(= 5번의 주고받음)의 메시지만 유지하는 슬라이딩 윈도우 방식 사용.
chat_memory = []
MAX_MEMORY = 10  # user + assistant 쌍으로 카운트 → 최대 5턴


def handle_user(text: str, model: str = "llama3.1:8b") -> str:
    """
    [핵심 메인 처리 함수]
    사용자 자연어 명령을 받아 LLM에 전달하고, tool_call 결과를 처리하여
    최종 한국어 응답 문장을 반환한다.

    처리 흐름:
        1. 사용자 입력 → chat_memory 추가
        2. system_prompt + chat_memory → LLM 전달
        3a. tool_call 없음: 일반 대화 응답 반환
        3b. tool_call 있음:
            - set_temperature: 범위(16~30°C) 검증
            - get_weather: fetch_weather() → confirmation_weather() 처리
            - 기타 제어: execute_tool_call() → confirmation_korean() 처리
        4. 응답을 chat_memory에 추가 + JSONL/CSV 로그 저장

    Args:
        text  (str): 사용자 자연어 입력
        model (str): 사용할 LLM 모델 이름 (Ollama에서 pull된 모델)

    Returns:
        str: 사용자에게 출력할 최종 응답 문장
    """
    global chat_memory

    # 1. 사용자 발화를 대화 메모리에 추가
    chat_memory.append({"role": "user", "content": text})

    # 2. system prompt + 누적 대화를 통째로 LLM에 전달 (Few-shot 형태)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + chat_memory

    # 로그 레코드 초기화
    base_record = {
        "timestamp":    datetime.now().isoformat(timespec="seconds"),
        "session_id":   SESSION_ID,
        "user_text":    text,
        "tool_calls":   [],
        "exec_results": [],
        "normal_reply": "",
        "model":        model,
    }

    try:
        # LLM API 호출 (Ollama via bore 터널)
        # temperature=0.2: 낮은 값으로 결정론적 tool_call 선택 유도
        # tool_choice="auto": LLM이 스스로 도구 사용 여부 판단
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.2,
        )
        msg = resp.choices[0].message

        # --- 분기 A: 도구 호출 없음 (OUT OF DOMAIN 또는 일반 대화) ---
        if not getattr(msg, "tool_calls", None):
            result_text = (msg.content or "요청을 이해했어요.").strip()
            base_record["normal_reply"] = result_text

            # 응답을 메모리에 추가 후 슬라이딩 윈도우 적용
            chat_memory.append({"role": "assistant", "content": result_text})
            if len(chat_memory) > MAX_MEMORY:
                chat_memory = chat_memory[-MAX_MEMORY:]  # 오래된 기억 삭제

            save_jsonl(base_record)
            append_csv(base_record)
            return result_text

        # --- 분기 B: 도구 호출 있음 (장비 제어 또는 날씨 조회) ---
        confirmations = []           # 각 tool_call의 한국어 응답 문장 누적
        tool_calls_serialized = []   # 로그용 직렬화 데이터
        exec_results_serialized = [] # 로그용 실행 결과 데이터

        # LLM이 복수의 tool_call을 동시에 요청할 수 있음 (예: 온도 + 팬 동시 설정)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")

            # [온도 범위 검증] JSON Schema 검증만으로 부족하므로 코드 레벨에서 이중 검증
            if name == "set_temperature":
                t = float(args.get("temp_c", 0))
                if not (16 <= t <= 30):
                    result_text = "온도는 16~30°C 사이로만 설정할 수 있어요."
                    chat_memory.append({"role": "assistant", "content": result_text})
                    return result_text

            tool_calls_serialized.append({"name": name, "args": args})

            # [날씨 조회 처리] 다른 제어 도구와 달리 반환값(JSON)을 파싱해야 하므로 분리 처리
            if name == "get_weather":
                weather_json = fetch_weather(
                    date=args.get("date", "today"),
                    time=args.get("time")
                )
                # ⚠️ 개선 포인트: intent 파라미터를 LLM args에서 가져오도록 수정 권장
                # 현재는 intent가 항상 기본값 "summary"로 고정됨
                weather_text = confirmation_weather(
                    weather_json,
                    intent=args.get("intent", "summary"),  # LLM이 결정한 intent 반영
                    target_date=args.get("date", "today")
                )
                confirmations.append(weather_text)
                exec_results_serialized.append({"name": name, "args": args, "ok": True})
                continue  # 날씨는 execute_tool_call 불필요 → 다음 tool_call로 넘어감

            # [일반 장비 제어 실행] execute_tool_call → status.json 갱신
            ok = execute_tool_call(name, args)
            exec_results_serialized.append({"name": name, "args": args, "ok": bool(ok)})

            if not ok:
                # 제어 실패 시 즉시 에러 메시지 반환 (추가 tool_call 처리 중단)
                result_text = "장비 제어 중 오류가 발생했어요."
                chat_memory.append({"role": "assistant", "content": result_text})
                return result_text

            confirmations.append(confirmation_korean(name, args))

        # 모든 tool_call 처리 완료 후 확인 문장 합쳐서 최종 응답 생성
        result_text = " ".join(confirmations).strip()

        # 제어 결과를 메모리에 추가 (다음 대화에서 "방금 뭐 했지?" 같은 질문 대응)
        chat_memory.append({"role": "assistant", "content": result_text})
        if len(chat_memory) > MAX_MEMORY:
            chat_memory = chat_memory[-MAX_MEMORY:]

        # 로그 레코드 완성 후 저장
        base_record["normal_reply"]   = result_text
        base_record["tool_calls"]     = tool_calls_serialized
        base_record["exec_results"]   = exec_results_serialized

        save_jsonl(base_record)
        append_csv(base_record)
        return result_text

    except Exception as e:
        # LLM 통신 오류 또는 예상치 못한 예외 처리
        # 실제 서비스에서는 더 구체적인 에러 메시지와 재시도 로직 추가 권장
        print(f"[ERROR] LLM 처리 중 오류 발생: {e}")
        return "오류가 발생했어요. 다시 시도해 주세요."


# ====== 실행 모드 분기 ======
# CLI(터미널) 실행 시: argparse로 --file 또는 --dir 인자를 받아 폴링 시작
# Jupyter Notebook 실행 시: COMMAND_FILE_PATH를 고정으로 사용해 자동 폴링 시작
if __name__ == "__main__" and not ("ipykernel" in sys.modules):
    # --- CLI 모드 ---
    import argparse

    parser = argparse.ArgumentParser(description="Excavator AC Controller (file/dir polling)")
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--file", type=str, help="단일 텍스트 파일 '마지막 줄' 폴링 실행")
    group.add_argument("--dir",  type=str, help="폴더 폴링 모드: 가장 최근 .txt의 '마지막 줄' 실행")
    parser.add_argument("--interval", type=float, default=1.0, help="폴링 간격(초), 기본 1.0")
    # ⚠️ 개선 포인트: --model 인자가 파싱되지만 handle_user()에 전달되지 않음
    # 수정 방법: loop_watch_file/loop_watch_directory 함수에 model 인자 추가 필요
    parser.add_argument("--model", type=str, default="llama3.1:8b", help="모델명 (handle_user에 전달)")
    args = parser.parse_args()

    if args.file:
        loop_watch_file(args.file, interval_sec=args.interval)
    else:
        loop_watch_directory(args.dir, interval_sec=args.interval)

else:
    # --- Jupyter Notebook 모드 ---
    # ipykernel이 로드된 환경에서는 자동으로 파일 폴링을 시작한다.
    print("[INFO] Jupyter environment detected → file polling started")
    loop_watch_file(COMMAND_FILE_PATH, interval_sec=1.0)
